In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score, confusion_matrix, 
    classification_report, roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)

# ==========================================
# TASK 1 & 2: DATA LOADING & ENCODING
# ==========================================
# Load cleaned dataset
df = pd.read_csv("cleaned_data.csv")

# Define target and continuous label
# NOTE: Replace 'Target_Continuous' with your actual continuous column name
y_reg = df['Target_Continuous'] 

# Create binary classification label using median split
y_clf = (y_reg > y_reg.median()).astype(int)

# Separate features matrix X
X = df.drop(columns=['Target_Continuous'])

# 1. Label Encoding for Ordinal Features (with a clear natural order)
# Example: Mapping 'Low', 'Medium', 'High' to 0, 1, 2
# NOTE: Replace 'Categorical_Ordinal' with your actual ordinal column name if present
if 'Categorical_Ordinal' in X.columns:
    ordinal_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
    X['Categorical_Ordinal'] = X['Categorical_Ordinal'].map(ordinal_mapping)

# 2. One-Hot Encoding for Nominal Features (no natural order)
# Drops first dummy column to avoid multicollinearity (the dummy variable trap)
nominal_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
X = pd.get_dummies(X, columns=nominal_cols, drop_first=True, dtype=int)

# ==========================================
# TASK 3: LEAK-FREE SPLIT & SCALING
# ==========================================
# Split data ensuring identical indices for both regression and classification targets
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

# Initialize and fit scaler ONLY on the training features to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easy feature mapping
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

# ==========================================
# TASK 4: REGRESSION MODELING (OLS vs RIDGE)
# ==========================================
print("--- TASK 4: REGRESSION MODELING ---")

# Ordinary Least Squares (OLS) Linear Regression
ols_model = LinearRegression()
ols_model.fit(X_train_scaled, y_reg_train)
y_pred_reg_ols = ols_model.predict(X_test_scaled)

ols_mse = mean_squared_error(y_reg_test, y_pred_reg_ols)
ols_r2 = r2_score(y_reg_test, y_pred_reg_ols)

print(f"OLS Linear Regression - MSE: {ols_mse:.4f}, R²: {ols_r2:.4f}")

# Map and print OLS coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': ols_model.coef_,
    'Abs_Coefficient': np.abs(ols_model.coef_)
}).sort_values(by='Abs_Coefficient', ascending=False)

print("\nTop 3 Most Impactful Features (OLS):")
print(coef_df.head(3).to_string(index=False))

# Ridge Regression Experiment (alpha=1.0)
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_reg_train)
y_pred_reg_ridge = ridge_model.predict(X_test_scaled)

ridge_mse = mean_squared_error(y_reg_test, y_pred_reg_ridge)
ridge_r2 = r2_score(y_reg_test, y_pred_reg_ridge)

print(f"\nRidge Regression - MSE: {ridge_mse:.4f}, R²: {ridge_r2:.4f}\n")

# ==========================================
# TASK 5: BINARY CLASSIFICATION MODELING
# ==========================================
print("--- TASK 5: CLASSIFICATION MODELING ---")

# Check class imbalance in training set
class_counts = y_clf_train.value_counts(normalize=True)
print("Training Class Distribution:")
print(y_clf_train.value_counts())

# Dynamically handle class imbalance if minor class is < 35%
clf_kwargs = {'max_iter': 1000, 'random_state': 42}
if class_counts.min() < 0.35:
    print("\n[INFO] Imbalance detected (<35%). Applying class_weight='balanced'.")
    clf_kwargs['class_weight'] = 'balanced'
else:
    print("\n[INFO] Class distribution is balanced (>=35%). Proceeding with default weights.")

# Train Baseline Logistic Regression (C=1.0)
log_reg_baseline = LogisticRegression(**clf_kwargs, C=1.0)
log_reg_baseline.fit(X_train_scaled, y_clf_train)

y_pred_clf_base = log_reg_baseline.predict(X_test_scaled)
y_prob_clf_base = log_reg_baseline.predict_proba(X_test_scaled)[:, 1]

# Evaluation Metrics
print("\nConfusion Matrix:")
print(confusion_matrix(y_clf_test, y_pred_clf_base))
print("\nClassification Report:")
print(classification_report(y_clf_test, y_pred_clf_base))

base_auc = roc_auc_score(y_clf_test, y_prob_clf_base)
print(f"Baseline (C=1.0) ROC-AUC: {base_auc:.4f}\n")

# Generate and Save ROC Curve Plot
fpr, tpr, _ = roc_curve(y_clf_test, y_prob_clf_base)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {base_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.savefig('roc_curve.png', dpi=300)
plt.close()

# ==========================================
# TASK 5b: DECISION-THRESHOLD SENSITIVITY
# ==========================================
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

for t in thresholds:
    y_pred_thresh = (y_prob_clf_base >= t).astype(int)
    p = precision_score(y_clf_test, y_pred_thresh, zero_division=0)
    r = recall_score(y_clf_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_clf_test, y_pred_thresh, zero_division=0)
    threshold_results.append([t, p, r, f1])

df_thresh = pd.DataFrame(threshold_results, columns=['Threshold', 'Precision', 'Recall', 'F1'])
print("Decision-Threshold Sensitivity Table:")
print(df_thresh.to_string(index=False))
print("\n")

# ==========================================
# TASK 6: REGULARIZATION EXPERIMENT
# ==========================================
print("--- TASK 6: REGULARIZATION EXPERIMENT ---")

# Train heavily regularized model (C=0.01)
log_reg_reg = LogisticRegression(**clf_kwargs, C=0.01)
log_reg_reg.fit(X_train_scaled, y_clf_train)

y_pred_clf_reg = log_reg_reg.predict(X_test_scaled)
y_prob_clf_reg = log_reg_reg.predict_proba(X_test_scaled)[:, 1]

# Collect comparison metrics
p_base = precision_score(y_clf_test, y_pred_clf_base)
r_base = recall_score(y_clf_test, y_pred_clf_base)
auc_base = base_auc

p_reg = precision_score(y_clf_test, y_pred_clf_reg)
r_reg = recall_score(y_clf_test, y_pred_clf_reg)
auc_reg = roc_auc_score(y_clf_test, y_prob_clf_reg)

# ==========================================
# TASK 7: BOOTSTRAP CONFIDENCE INTERVAL
# ==========================================
np.random.seed(42)
n_bootstraps = 500
bootstrapped_auc_diffs = []

y_clf_test_arr = np.array(y_clf_test)

for _ in range(n_bootstraps):
    # Sample row indices with replacement
    indices = np.random.choice(len(y_clf_test_arr), size=len(y_clf_test_arr), replace=True)
    
    # Check if bootstrap sample contains both classes to avoid roc_auc error
    if len(np.unique(y_clf_test_arr[indices])) < 2:
        continue
        
    # Calculate AUC for both models on this sample
    auc_c10 = roc_auc_score(y_clf_test_arr[indices], y_prob_clf_base[indices])
    auc_c001 = roc_auc_score(y_clf_test_arr[indices], y_prob_clf_reg[indices])
    
    # Difference: Baseline (C=1.0) minus Regularized (C=0.01)
    bootstrapped_auc_diffs.append(auc_c10 - auc_c001)

mean_diff = np.mean(bootstrapped_auc_diffs)
ci_lower = np.percentile(bootstrapped_auc_diffs, 2.5)
ci_upper = np.percentile(bootstrapped_auc_diffs, 97.5)

print(f"Bootstrap Results (n={n_bootstraps}):")
print(f"Mean AUC Difference: {mean_diff:.4f}")
print(f"95% Confidence Interval: [{ci_lower:.4f}, {ci_upper:.4f}]")